# Diabetes Prediction Using Machine Learning



## 1. Import Required Libraries




In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

try:
    import matplotlib.pyplot as plt
except ImportError as error:
    plt = None
    print(f"matplotlib is not installed yet: {error}")

try:
    import seaborn as sns
except ImportError as error:
    sns = None
    print(f"seaborn is not installed yet: {error}")

try:
    import sklearn
except ImportError as error:
    sklearn = None
    print(f"scikit-learn is not installed yet: {error}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 2. Load Dataset Using pathlib



In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_RELATIVE_PATH = Path("data") / "raw" / "diabetes.csv"
OUTPUT_TABLES_RELATIVE_DIR = Path("outputs") / "tables" / "week_05"

DATASET_PATH = PROJECT_ROOT / DATASET_RELATIVE_PATH
OUTPUT_TABLES_DIR = PROJECT_ROOT / OUTPUT_TABLES_RELATIVE_DIR

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset file not found: {DATASET_RELATIVE_PATH}")

df = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.")
print(f"Dataset path: {DATASET_RELATIVE_PATH}")


Dataset loaded successfully.
Dataset path: data\raw\diabetes.csv


## 3. Verify Expected Columns



In [3]:
EXPECTED_COLUMNS = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]

missing_columns = [column for column in EXPECTED_COLUMNS if column not in df.columns]
extra_columns = [column for column in df.columns if column not in EXPECTED_COLUMNS]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

if extra_columns:
    print(f"Extra columns found: {extra_columns}")
else:
    print("All expected columns are present.")


All expected columns are present.


## 4. Dataset Shape and Sample Records




In [17]:
print(f"Dataset shape: {df.shape}")

print("First five records:")
display(df.head())

print("Last five records:")
display(df.tail())


Dataset shape: (768, 9)
First five records:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


Last five records:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1
767,1,93,70,31,0,30.4,0.315,23,0


## 5. Separate Input Features and Target Variable



In [4]:
target_variable = "Outcome"
X = df.drop(columns=target_variable)
y = df[target_variable]

print(f"Input feature matrix X shape: {X.shape}")
print(f"Target variable y shape: {y.shape}")
print("Input features:", list(X.columns))
print("Target variable:", target_variable)


Input feature matrix X shape: (768, 8)
Target variable y shape: (768,)
Input features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Target variable: Outcome


## 7. Detailed Feature Analysis

This table explains each feature and calculates its range, average values, zero counts, and unique-value counts. These values are calculated from the raw CSV file.


In [5]:
feature_descriptions = {
    "Pregnancies": "Number of times the patient has been pregnant",
    "Glucose": "Plasma glucose concentration from an oral glucose tolerance test",
    "BloodPressure": "Diastolic blood pressure measurement",
    "SkinThickness": "Triceps skin fold thickness measurement",
    "Insulin": "Two-hour serum insulin measurement",
    "BMI": "Body mass index calculated from height and weight",
    "DiabetesPedigreeFunction": "Score that represents diabetes family history influence",
    "Age": "Age of the patient in years",
    "Outcome": "Diabetes class label where 0 means non-diabetic and 1 means diabetic",
}

feature_analysis_rows = []
for column in df.columns:
    series = df[column]
    feature_analysis_rows.append({
        "feature_name": column,
        "description": feature_descriptions[column],
        "role": "Target" if column == target_variable else "Input",
        "data_type": str(series.dtype),
        "minimum_value": series.min(),
        "maximum_value": series.max(),
        "mean": round(series.mean(), 4),
        "median": round(series.median(), 4),
        "standard_deviation": round(series.std(), 4),
        "zero_value_count": int((series == 0).sum()),
        "unique_value_count": int(series.nunique()),
    })

feature_analysis = pd.DataFrame(feature_analysis_rows)
display(feature_analysis)


,feature_name,description,role,data_type,minimum_value,maximum_value,mean,median,standard_deviation,zero_value_count,unique_value_count
0,Pregnancies,Number of times the patient has been pregnant,Input,int64,0.000,17.00,3.8451,3.0000,3.3696,111,17
1,Glucose,Plasma glucose concentration from an oral gluc...,Input,int64,0.000,199.00,120.8945,117.0000,31.9726,5,136
2,BloodPressure,Diastolic blood pressure measurement,Input,int64,0.000,122.00,69.1055,72.0000,19.3558,35,47
3,SkinThickness,Triceps skin fold thickness measurement,Input,int64,0.000,99.00,20.5365,23.0000,15.9522,227,51
4,Insulin,Two-hour serum insulin measurement,Input,int64,0.000,846.00,79.7995,30.5000,115.2440,374,186
5,BMI,Body mass index calculated from height and weight,Input,float64,0.000,67.10,31.9926,32.0000,7.8842,11,248
6,DiabetesPedigreeFunction,Score that represents diabetes family history ...,Input,float64,0.078,2.42,0.4719,0.3725,0.3313,0,517
7,Age,Age of the patient in years,Input,int64,21.000,81.00,33.2409,29.0000,11.7602,0,52
8,Outcome,Diabetes class label where 0 means non-diabeti...,Target,int64,0.000,1.00,0.3490,0.0000,0.4770,500,2


## 8. Target Variable Analysis

The target distribution shows class labels, class counts, and percentages. A dataset is considered imbalanced here if the minority class is below 40% of records.


In [8]:
target_distribution = (
    y.value_counts()
    .sort_index()
    .rename_axis("class_label")
    .reset_index(name="count")
)
target_distribution["percentage"] = (target_distribution["count"] / len(y) * 100).round(2)

minority_percentage = target_distribution["percentage"].min()
balance_status = "Moderately imbalanced" if minority_percentage < 40 else "Relatively balanced"
target_distribution["balance_note"] = balance_status

display(target_distribution)
print(f"Class balance observation: {balance_status}")



,class_label,count,percentage,balance_note
0,0,500,65.1,Moderately imbalanced
1,1,268,34.9,Moderately imbalanced


Class balance observation: Moderately imbalanced


## 9. Value Range Check for All Columns



In [9]:
range_check = pd.DataFrame({
    "column_name": df.columns,
    "minimum_value": [df[column].min() for column in df.columns],
    "maximum_value": [df[column].max() for column in df.columns],
    "zero_value_count": [int((df[column] == 0).sum()) for column in df.columns],
    "unique_value_count": [int(df[column].nunique()) for column in df.columns],
})

display(range_check)


,column_name,minimum_value,maximum_value,zero_value_count,unique_value_count
0,Pregnancies,0.000,17.00,111,17
1,Glucose,0.000,199.00,5,136
2,BloodPressure,0.000,122.00,35,47
3,SkinThickness,0.000,99.00,227,51
4,Insulin,0.000,846.00,374,186
5,BMI,0.000,67.10,11,248
6,DiabetesPedigreeFunction,0.078,2.42,0,517
7,Age,21.000,81.00,0,52
8,Outcome,0.000,1.00,500,2


## 10. Suspicious Zero Values

Zero values in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` may represent missing or invalid medical measurements. For example, a glucose value of zero, blood pressure of zero, or BMI of zero is not medically realistic for a normal patient record. Skin thickness and insulin values of zero may also indicate that the measurement was not recorded.




In [11]:
suspicious_zero_columns = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

suspicious_zero_summary = pd.DataFrame({
    "feature_name": suspicious_zero_columns,
    "zero_value_count": [int((df[column] == 0).sum()) for column in suspicious_zero_columns],
    "reason_for_suspicion": [
        "Glucose value of zero is not medically realistic for a living patient measurement.",
        "Blood pressure value of zero is not medically realistic for a normal patient record.",
        "Skin thickness value of zero may indicate the measurement was not recorded.",
        "Insulin value of zero may indicate the measurement was not recorded.",
        "BMI value of zero is not medically realistic for a patient record.",
    ],
 
})

display(suspicious_zero_summary)


,feature_name,zero_value_count,reason_for_suspicion
0,Glucose,5,Glucose value of zero is not medically realist...
1,BloodPressure,35,Blood pressure value of zero is not medically ...
2,SkinThickness,227,Skin thickness value of zero may indicate the ...
3,Insulin,374,Insulin value of zero may indicate the measure...
4,BMI,11,BMI value of zero is not medically realistic f...


## 12. Save Tables




In [13]:
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

tables_to_save = {
    "feature_analysis.csv": feature_analysis,
    "target_distribution.csv": target_distribution,
    "range_check.csv": range_check,
    "suspicious_zero_values.csv": suspicious_zero_summary,
   
}

for file_name, table in tables_to_save.items():
    table.to_csv(OUTPUT_TABLES_DIR / file_name, index=False)

print("Saved Week 5 tables:")
for file_name in tables_to_save:
    print(f"- {OUTPUT_TABLES_RELATIVE_DIR / file_name}")


Saved Week 5 tables:
- outputs\tables\week_05\feature_analysis.csv
- outputs\tables\week_05\target_distribution.csv
- outputs\tables\week_05\range_check.csv
- outputs\tables\week_05\suspicious_zero_values.csv
